# Football Match Prediction Model Training

This notebook walks through training machine learning models to predict football match outcomes. We'll train three models:
- **Match Result**: Home Win, Draw, or Away Win
- **Over/Under 2.5 Goals**: Whether total goals will be over or under 2.5
- **Both Teams to Score (BTTS)**: Whether both teams will score

We'll use Python, pandas, scikit-learn, and joblib. At the end, models are saved as `.pkl` files for the Streamlit app.

In [58]:
import pandas as pd
import numpy as np
import os
import joblib


## Step 1: Load the Data

We load the engineered dataset that contains match features and target labels.

In [59]:
# Load the engineered dataset
df = pd.read_csv(r"C:\Users\pc\Downloads\dataset_features.csv")
raw = pd.read_csv(r"C:\Users\pc\Downloads\E0_Cleaned.csv") 
df['Date'] = pd.to_datetime(df['Date'])
raw['Date'] = pd.to_datetime(raw['Date'])
df = df.merge(
    raw[['HomeTeam', 'AwayTeam', 'Date', 'FTHG', 'FTAG']],
    on=['HomeTeam', 'AwayTeam', 'Date'],
    how='left'
)
print('Shape:', df.shape)
print('Columns:', list(df.columns)[:10], '...')
print(df.head())
print(df['FTHG'].isna().sum())
df = df.dropna(subset=['FTHG', 'FTAG'])

Shape: (330, 23)
Columns: ['HomeTeam', 'AwayTeam', 'Date', 'home_avg_goals', 'home_avg_shots', 'home_avg_conceded', 'home_good_matches', 'home_is_offensive', 'away_avg_goals', 'away_avg_shots'] ...
        HomeTeam     AwayTeam       Date  home_avg_goals  home_avg_shots  \
0      Tottenham       Wolves 2025-09-27             2.0             4.0   
1       Man City      Burnley 2025-09-27             1.8             4.0   
2          Leeds  Bournemouth 2025-09-27             0.8             2.4   
3  Nott'm Forest   Sunderland 2025-09-27             1.0             3.6   
4        Chelsea     Brighton 2025-09-27             2.0             4.6   

   home_avg_conceded  home_good_matches  home_is_offensive  away_avg_goals  \
0                0.6                  4                  1             0.6   
1                1.0                  2                  1             1.0   
2                1.4                  2                  0             1.2   
3                1.8             

## Step 2: Explore Target Variables

The dataset has a `Target` column for match result (H=Home Win, D=Draw, A=Away Win). For other tasks, we'll create binary targets from the original match data.

In [60]:
# Check target distribution
target_counts = df['Target'].value_counts()
print('Target distribution:')
display(target_counts)
print('Class names: H=Home Win, D=Draw, A=Away Win')

Target distribution:


Target
H    139
A    101
D     90
Name: count, dtype: int64

Class names: H=Home Win, D=Draw, A=Away Win


## Step 3: Prepare Features (X) and Target (y)

We'll use the 19 feature columns (excluding Team names, Date, and Target).

In [61]:
# Separate features and target dynamically
cols_to_drop = ['HomeTeam', 'AwayTeam', 'Date', 'Target', 'FTHG', 'FTAG']
X = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
y = df['Target']
print('Features shape:', X.shape)
print('Target shape:', y.shape)
print(X.columns.tolist())

Features shape: (330, 17)
Target shape: (330,)
['home_avg_goals', 'home_avg_shots', 'home_avg_conceded', 'home_good_matches', 'home_is_offensive', 'away_avg_goals', 'away_avg_shots', 'away_avg_conceded', 'away_good_matches', 'away_is_offensive', 'diff_avg_goals', 'diff_avg_shots', 'diff_avg_conceded', 'diff_good_matches', 'h2h_home_wr', 'h2h_draw_wr', 'h2h_away_wr']


## Step 4: Train-Test Split

We split the data into training (80%) and testing (20%) sets. This ensures the model generalizes to unseen matches.

In [62]:
from sklearn.model_selection import train_test_split

# Split with random state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Training set size:', X_train.shape[0])
print('Test set size:', X_test.shape[0])
print(y_train.value_counts())
print(y_test.value_counts())

Training set size: 264
Test set size: 66
Target
H    111
A     81
D     72
Name: count, dtype: int64
Target
H    28
A    20
D    18
Name: count, dtype: int64


## Step 5: Scale the Features

Gradient Boosting works well with scaled data. We'll use StandardScaler to standardize features (mean=0, std=1).

In [63]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Fit on training data, transform both train and test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)
print('Scaling complete. Mean after scaling:', X_train_scaled.mean().round(4).tolist())

Scaling complete. Mean after scaling: [0.0, 0.0, -0.0, 0.0, 0.0, 0.0, 0.0, -0.0, -0.0, 0.0, 0.0, 0.0, -0.0, 0.0, 0.0, -0.0, 0.0]


## Step 6: Train the Match Result Model

We train a Gradient Boosting Classifier to predict Home Win (H), Draw (D), or Away Win (A).

In [64]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize model
model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Train on scaled training data
model.fit(X_train_scaled, y_train)

# Predict on test set
y_pred = model.predict(X_test_scaled)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print('Test Accuracy:', round(accuracy, 4))
print(classification_report(y_test, y_pred))

Test Accuracy: 0.3636
              precision    recall  f1-score   support

           A       0.29      0.25      0.27        20
           D       0.00      0.00      0.00        18
           H       0.47      0.68      0.56        28

    accuracy                           0.36        66
   macro avg       0.26      0.31      0.28        66
weighted avg       0.29      0.36      0.32        66



## Step 7: Train Over/Under 2.5 Goals Model

For this binary classification, we create a target: 1 = Over 2.5 goals, 0 = Under 2.5 goals.

In [65]:
# Create binary target for over/under 2.5
y_ou = (df['FTHG'] + df['FTAG'] > 2).astype(int)  # 1 if over 2.5, 0 if under
print('Over/Under target distribution:')
print(y_ou.value_counts())

# Split
X_train, X_test, y_train_ou, y_test_ou = train_test_split(
    X, y_ou, test_size=0.2, random_state=42, stratify=y_ou
)

# Scale (using same scaler or new one - here we use a new scaler for clarity)
scaler_ou = StandardScaler()
X_train_ou_scaled = scaler_ou.fit_transform(X_train)
X_test_ou_scaled = scaler_ou.transform(X_test)

# Train model
model_ou = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
model_ou.fit(X_train_ou_scaled, y_train_ou)

# Predict and evaluate
y_ou_pred = model_ou.predict(X_test_ou_scaled)
accuracy_ou = accuracy_score(y_test_ou, y_ou_pred)
print('Over/Under 2.5 Accuracy:', round(accuracy_ou, 4))

Over/Under target distribution:
1    184
0    146
Name: count, dtype: int64
Over/Under 2.5 Accuracy: 0.4242


## Step 8: Train BTTS (Both Teams to Score) Model

Binary target: 1 = Both teams score, 0 = At least one team doesn't score.

In [66]:
# Create BTTS target: 1 if both teams scored, 0 otherwise
y_btts = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).astype(int)
print('BTTS target distribution:')
print(y_btts.value_counts())

# Split
X_train, X_test, y_train_btts, y_test_btts = train_test_split(
    X, y_btts, test_size=0.2, random_state=42, stratify=y_btts
)

# Scale
scaler_btts = StandardScaler()
X_train_btts_scaled = scaler_btts.fit_transform(X_train)
X_test_btts_scaled = scaler_btts.transform(X_test)

# Train model
model_btts = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
model_btts.fit(X_train_btts_scaled, y_train_btts)

# Predict and evaluate
y_btts_pred = model_btts.predict(X_test_btts_scaled)
accuracy_btts = accuracy_score(y_test_btts, y_btts_pred)
print('BTTS Accuracy:', round(accuracy_btts, 4))

BTTS target distribution:
1    191
0    139
Name: count, dtype: int64
BTTS Accuracy: 0.3788


## Step 9: Save Models and Preprocessors

Now we save all trained models, scalers, and feature names using joblib. This is exactly what the Streamlit app uses for predictions.

In [67]:
import joblib

# Save result model and companions
joblib.dump(model, 'models/best_match_predictor_model.pkl')
joblib.dump(scaler, 'models/scaler_result.pkl')
joblib.dump(list(X.columns), 'models/feature_names_result.pkl')
joblib.dump('placeholder', 'models/label_encoder_result.pkl')

# Save Over/Under model and companions
joblib.dump(model_ou, 'models/over_under_model.pkl')
joblib.dump(scaler_ou, 'models/scaler_ou.pkl')

# Save BTTS model and companions
joblib.dump(model_btts, 'models/btts_model.pkl')
joblib.dump(scaler_btts, 'models/scaler_btts.pkl')
joblib.dump(list(X.columns), 'models/feature_names_btts.pkl')

# Verify files were saved
model_files = [f for f in os.listdir('models/') if f.endswith('.pkl')]
print('Saved model files:')
print(model_files)

Saved model files:
['best_match_predictor_model.pkl', 'btts_model.pkl', 'feature_names_btts.pkl', 'feature_names_ou.pkl', 'feature_names_result.pkl', 'label_encoder_result.pkl', 'over_under_model.pkl', 'scaler_btts.pkl', 'scaler_ou.pkl', 'scaler_result.pkl']


## Step 10: Load and Test a Saved Model (Optional)

You can load a saved model and make a prediction on new data, just like the Streamlit app does.

In [68]:
# Example: Load the result model and make a prediction
# from utils import load_models
# models = load_models()
# model_res, le_res, scaler_res, feat_names_res = models['result']
# print('Model loaded successfully!')
# print('Features:', feat_names_res[:5], '...')
# print('Classes:', le_res.classes_)

## Training Complete!

All three models are now trained and saved. The Streamlit app (app.py) loads these .pkl files to make live predictions on match data.

**Next steps:**
- Run `streamlit run app.py` to use the prediction app
- Add more features (shots on target, possession, etc.) for better accuracy
- Train on more historical data for improved performance